# ShqipAI: Maximum Quality E4B Fine-Tuning with Unsloth

**Model**: Gemma 4 E4B (Effective 4B parameters)

**Hardware**: GPU T4 x2 (30 hours available) - **Unsloth works great on T4!**

**Time**: ~4 hours (Unsloth is 2x faster than standard training)

**Goal**: Create the smartest educational tutor that runs on budget laptops

---

## Why Unsloth?
- **2x faster** training
- **70% less memory** usage
- Works perfectly on **T4 GPU**
- Supports Gemma 4 E4B natively
- Easy GGUF export for Ollama

---

**IMPORTANT**: Set Accelerator to **GPU T4 x2** in Settings!

## Step 1: Install Unsloth (Fast Version)

In [ ]:
# FAST INSTALL - No xformers build (takes ~1 minute)
# Skip xformers on Kaggle - Unsloth works fine without it

import subprocess
import sys

# Install Unsloth core
subprocess.run([
    sys.executable, '-m', 'pip', 'install', 
    'unsloth', 
    '--quiet', '--no-warn-script-level'
], check=True)

# Install dependencies (pre-built, no compilation)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', 
    'trl', 'peft', 'accelerate', 'bitsandbytes',
    '--quiet', '--no-warn-script-level'
], check=True)

print('Installation complete!')

# Verify
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 2: Hugging Face Login

In [ ]:
from huggingface_hub import login

# PASTE YOUR TOKEN HERE - Get it from: https://huggingface.co/settings/tokens
HF_TOKEN = 'YOUR_HUGGING_FACE_TOKEN_HERE'

login(token=HF_TOKEN)
print('Logged in to Hugging Face!')

## Step 3: Load Gemma 4 E4B with Unsloth

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = 'google/gemma-4-e4b-it'  # Gemma 4 E4B

print(f'Loading {MODEL_NAME} with Unsloth...')
print('This takes 2-3 minutes, please wait...')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,        # Full context for tutoring
    dtype=None,                  # Auto-detect (bfloat16 on T4)
    load_in_4bit=True,           # 4-bit quantization for memory efficiency
    token=HF_TOKEN,
)

print(f'Model loaded! Parameters: {model.num_parameters()/1e9:.1f}B')

## Step 4: Add LoRA Adapters (Maximum Quality)

In [ ]:
# Add LoRA adapters with maximum capacity
model = FastLanguageModel.get_peft_model(
    model,
    r=64,                        # Maximum rank for best quality
    lora_alpha=128,              # 2x rank for stable training
    lora_dropout=0.05,           # Light regularization
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    bias='none',
    use_gradient_checkpointing='unsloth',  # Unsloth optimized
    random_state=42,
)

print('LoRA adapters added with maximum quality settings!')
model.print_trainable_parameters()

## Step 5: Load Training Data

In [ ]:
from datasets import load_dataset
import glob

# Find uploaded data file
data_files = glob.glob('/kaggle/input/**/*.jsonl', recursive=True)
print(f'Found data files: {data_files}')

if data_files:
    DATA_PATH = data_files[0]
    print(f'Using: {DATA_PATH}')
    
    dataset = load_dataset('json', data_files=DATA_PATH, split='train')
    print(f'Loaded {len(dataset)} training examples')
    
    # Show sample
    print('\nSample example:')
    print(dataset[0]['text'][:500] + '...')
else:
    print('ERROR: No data file found!')
    print('Please upload your educational_data.jsonl from Phase 1')

## Step 6: Configure Training (Maximum Quality)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Calculate training info
BATCH_SIZE = 2
GRAD_ACCUM = 8
EPOCHS = 5
effective_batch = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = len(dataset) // effective_batch
total_steps = steps_per_epoch * EPOCHS

print('='*60)
print('TRAINING CONFIGURATION (Maximum Quality with Unsloth)')
print('='*60)
print(f'Model: Gemma 4 E4B')
print(f'Training examples: {len(dataset)}')
print(f'Epochs: {EPOCHS}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Gradient accumulation: {GRAD_ACCUM}')
print(f'Effective batch: {effective_batch}')
print(f'Steps per epoch: {steps_per_epoch}')
print(f'Total training steps: {total_steps}')
print(f'Estimated time: ~{total_steps * 1 // 60} hours (Unsloth is 2x faster!)')
print('='*60)

In [ ]:
# Create trainer with Unsloth optimizations
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_ratio=0.1,
        num_train_epochs=EPOCHS,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        output_dir='./outputs',
        report_to='none',
        save_strategy='steps',
        save_steps=200,
        save_total_limit=3,
    ),
)

print('Trainer configured!')

## Step 7: START TRAINING!

**This will take ~4 hours with Unsloth.**

**Watch the loss:**
- Starts around 1.5-2.0
- Should decrease to 0.3-0.5

**DO NOT CLOSE THIS TAB!**

In [ ]:
print('\n' + '='*60)
print('STARTING TRAINING')
print('='*60)
print(f'Model: Gemma 4 E4B')
print(f'Training examples: {len(dataset)}')
print(f'Epochs: {EPOCHS}')
print(f'Estimated time: ~4 hours')
print('')
print('Watch the loss decrease from ~1.5 to ~0.3')
print('DO NOT CLOSE THIS TAB!')
print('='*60 + '\n')

# START TRAINING
trainer.train()

print('\n' + '='*60)
print('TRAINING COMPLETE!')
print('='*60)

## Step 8: Save as GGUF for Ollama

In [ ]:
# Save directly as GGUF (Unsloth feature!)
print('Saving as GGUF for Ollama...')

model.save_pretrained_gguf(
    'shqipai-tutor-e4b',
    tokenizer,
    quantization_method='q4_k_m',
)

print('GGUF saved!')
!ls -la shqipai-tutor-e4b/

## Step 9: Test the Fine-Tuned Model

In [ ]:
# Switch to inference mode
FastLanguageModel.for_inference(model)

# Test prompts
test_prompts = [
    'Explain photosynthesis to a 10-year-old student.',
    'How do I solve 2x + 5 = 13?',
]

print('Testing fine-tuned model...\n')

for i, prompt in enumerate(test_prompts):
    print(f'--- Test {i+1} ---')
    print(f'Q: {prompt}')
    
    inputs = tokenizer(
        [f'<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n'],
        return_tensors='pt'
    ).to('cuda')
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        use_cache=True,
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if 'model' in response:
        response = response.split('model')[-1].strip()
    print(f'A: {response}')
    print()

## Step 10: Download Your Model

**Download from Output panel:**
- `shqipai-tutor-e4b/unsloth.Q4_K_M.gguf` - Ready for Ollama!

---

## Use with Ollama

```bash
ollama create shqipai-tutor -f Modelfile
ollama run shqipai-tutor
```